# SEM image → shape adaptive grinding, ductile and brittle

**Shape adaptive grinding (SAG)** replaces the rigid wheel with a *compliant*
one: a stiff hub, a polyurethane layer a few millimetres thick, and an abrasive
pad on the outside. Press it against the work and the layer squashes, so line
contact spreads into an **area**.

That single fact is the whole process. The contact load is shared by every
grain the patch covers — hundreds of thousands of them — so the force on each
one collapses to $10^{-5}$ N and the depth each takes collapses with it. A
material that fractures under a conventional wheel can then be removed by
**plastic flow**, which is how a brittle cermet reaches a 21 nm finish.

### What this notebook does

You give it SEM micrographs of your abrasive. It measures every grain,
reconstructs each as a 3-D solid, solves the compliant contact, and writes two
Abaqus decks:

| deck | question it answers | resolves $d_c$? |
|---|---|---|
| **MACRO** | the *contact* — patch size, pressure, engaged grains, load per grain | no |
| **MICRO** | the *transition* — SDV13, ductile against brittle | **yes**, at $d_c/5$ |

They are coupled by one number: the per-grain load MACRO computes is what MICRO
applies. Both decks print it, so the pair cannot be quoted out of step.

### How the transition is decided

The other two notebooks in this project compare a *prescribed* chip thickness
$h(u)$ against $d_c$. That needs a known trajectory. Here there isn't one — with
a compliant tool the load per grain is the *answer*, not an input. So SAG uses
the **local energy criterion**:

$$W_p \cdot L_c \;\ge\; \Psi\,\frac{K_c^2}{E}$$

accumulated plastic work per unit volume, times the element's own length,
against a fracture energy. It needs no geometry, and it triggers on **history**
— a point starts ductile and turns brittle as work accumulates under repeated
grain passes, which is what a polishing pad physically does.

With $\Psi = 0$ the subroutine derives $\Psi = d_c E H/K_c^2$, making the
threshold exactly $W_p L_c \ge H d_c$. So a **measured** $d_c$ carries straight
through with no new calibration.

> **One property to know before quoting a result.** The criterion is
> regularised by $L_c$, so it is mesh-dependent *by construction*: halving the
> element halves the work density needed to trigger. That is correct for a
> fracture-energy criterion, and it means $\Psi$ is calibrated **for a mesh**.
> Every deck states its element size. Cell 10 measures the sensitivity.

### Reference

Ghosh, Sidpara & Bandyopadhyay (2021), *Brittle-ductile transition in compliant
finishing of HVOF sprayed hard WC-Co coating*, Int. J. Refractory Metals and
Hard Materials **99**, 105610. The contact chain in cell 4 is that paper's
eqs. 1–16, and cell 11 rebuilds its experiment.

In [ ]:
#@title 1 - Setup: unpack the pipeline (run once) { display-mode: "form" }
# semgrit (including the four new SAG modules), semgrit_multi, both VUMATs and
# every gate are embedded below, so this notebook is self-contained.
import base64, gzip, io, os, subprocess, sys, tarfile

PAYLOAD = (
    "..."
)

WORK = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(WORK)
with tarfile.open(fileobj=io.BytesIO(gzip.decompress(base64.b64decode(PAYLOAD)))) as tf:
    tf.extractall(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

missing = []
for mod, pip in [("numpy", "numpy"), ("scipy", "scipy"),
                 ("skimage", "scikit-image"),
                 ("cv2", "opencv-python-headless"), ("shapely", "shapely"),
                 ("PIL", "pillow"), ("mapbox_earcut", "mapbox-earcut"),
                 ("matplotlib", "matplotlib")]:
    try:
        __import__(mod)
    except ImportError:
        missing.append(pip)
if missing:
    print("installing:", " ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *missing],
                   check=True)

from semgrit import sag as _sag
from semgrit import sagdeck as _sd
from semgrit import sagemit as _se       # noqa: F401
from semgrit import meshview as _mv      # noqa: F401
print("pipeline ready in", WORK)
print("SAG modules : sag (contact), sagdeck (planner), sagwrite + sagemit")
print("              (deformable-tool decks), meshview (mesh in the viewer)")
print("subroutine  : vumat_grind2.for -- 58 constants, energy criterion")


def need(names, where):
    """Stop with the cell to run, instead of a NameError on a stray name."""
    absent = [n for n in names.split() if n not in globals()]
    if absent:
        raise SystemExit("run %s first - this cell needs %s"
                         % (where, ", ".join(absent)))

## 2 · Your abrasive pad, under the microscope

SAG pads are characterised by two numbers the contact model needs: the **grain
size** $d_g$ and the **areal density** $C_0$ of grains on the pad. Both come
from SEM micrographs of the pad itself.

Upload your own images, or leave the default to use the B4C micrographs
embedded in this notebook.

In [ ]:
#@title 2 - Where are your SEM images? { display-mode: "form" }
SOURCE = "bundled"  #@param ["bundled", "upload", "google drive", "already on disk"]
IMAGE_PATH = "/content/drive/MyDrive/sem/*.tif"  #@param {type:"string"}
PIXEL_SIZE_UM = 0.0  #@param {type:"number"}
#@markdown `PIXEL_SIZE_UM = 0` reads the scale from the SEM databar.
import glob, os

if SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    IMAGES = sorted(os.path.join(os.getcwd(), n) for n in up)
elif SOURCE == "google drive":
    from google.colab import drive
    drive.mount("/content/drive")
    IMAGES = sorted(glob.glob(IMAGE_PATH))
elif SOURCE == "bundled":
    IMAGES = sorted(glob.glob(os.path.join(WORK, "B4C_1*.tif")))
    if not IMAGES:
        raise SystemExit("no bundled images found; choose 'upload' instead")
else:
    IMAGES = sorted(glob.glob(IMAGE_PATH))

if not IMAGES:
    raise SystemExit("no images matched %r" % IMAGE_PATH)
print("%d image(s):" % len(IMAGES))
for p in IMAGES:
    print("   ", os.path.basename(p))

## 3 · Measure the grains

Every grain is segmented, measured (25 shape descriptors), and reconstructed as
a watertight 3-D solid whose maximum projected cross-section **is** the measured
outline. The figures below show every stage, so nothing is taken on trust.

In [ ]:
#@title 3 - Measure every grain, and show the work { display-mode: "form" }
SHOW_STAGES = True   #@param {type:"boolean"}
need("IMAGES", "cell 2")
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["figure.dpi"] = 110
_show = plt.show

from semgrit import figures as figs
from semgrit.quick import measure_images

MEAS = measure_images(IMAGES, os.path.join(WORK, "_sag_meas"),
                      pixel_size_um=(PIXEL_SIZE_UM or None),
                      keep_stages=SHOW_STAGES, log=print)
SOLIDS = MEAS["solids"]
GRAINS = MEAS["grains"]
print("")
print("%d grain solids from %d image(s)" % (len(SOLIDS), len(IMAGES)))
hs = [s.height_um for s in SOLIDS]
print("heights %.2f to %.2f um (mean %.2f)"
      % (min(hs), max(hs), sum(hs) / len(hs)))

if SHOW_STAGES and MEAS.get("per_image"):
    rec = MEAS["per_image"][0]
    for fn in (figs.calibration, figs.segmentation_stages,
               figs.segmentation_overlay, figs.outline_fidelity,
               figs.solid_verification):
        try:
            fn(rec)
            _show()
        except Exception as exc:
            print("(%s skipped: %s)" % (fn.__name__, exc))
    figs.measurement_distributions(GRAINS)
    _show()
    figs.grain_gallery(SOLIDS)
    _show()

## 4 · The compliant contact

Now the SAG-specific physics, following the reference paper's eqs. 1–16.

The tool is pressed in by the **wheel compression** $T$, and Hertz gives the
load:

$$F_N = 1.44\,E_{eq}\,R^{1/2}\,T^{3/2}
\qquad
E_{eq} = \left(\frac{1-\nu_w^2}{E_w} + \frac{1-\nu_t^2}{E_t}\right)^{-1}$$

The patch area and length are empirical fits to measured finishing spots:

$$A_s = 138.22\,T^{0.151}N^{0.009}
\qquad
L_s = 17.69\,T^{0.232}N^{0.012}$$

The load is then divided among the grains the patch covers, and each grain's
indentation follows from the Brinell relation:

$$N_{abr} = C_a A_s
\qquad
F_n = \frac{F_N}{N_{abr}}
\qquad
d = \frac{d_g}{2} - \tfrac{1}{2}\sqrt{d_g^2 - d_i^2}$$

**Set your process here.** Everything downstream — patch size, per-grain load,
mesh, deck size, runtime — follows from these numbers.

In [ ]:
#@title 4 - Your SAG process { display-mode: "form" }
#@markdown ### The tool
WHEEL_DIAMETER_MM = 125.0   #@param {type:"number"}
WHEEL_WIDTH_MM = 10.0       #@param {type:"number"}
LAYER_THICKNESS_MM = 5.0    #@param {type:"number"}
#@markdown Polyurethane, neo-Hookean. `E = 6*C10`, so C10 = 0.16606 is ~1.0 MPa.
PU_C10_MPA = 0.16606        #@param {type:"number"}
PU_DENSITY_KG_M3 = 1100.0   #@param {type:"number"}
PU_PRONY_G = 0.11           #@param {type:"number"}
PU_PRONY_TAU_S = 0.01       #@param {type:"number"}

#@markdown ### The process
COMPRESSION_MM = 0.4        #@param {type:"number"}
SPEED_RPM = 1050.0          #@param {type:"number"}
FRICTION = 0.2              #@param {type:"number"}
GRAIN_UM = 6.0              #@param [6.0, 15.0, 30.0] {type:"raw", allow-input: true}
#@markdown Pad density in grains/mm2. 0 uses the measured value for 6/15/30 um.
PAD_DENSITY_PER_MM2 = 0.0   #@param {type:"number"}

#@markdown ### The workpiece
MATERIAL = "wc_co"          #@param ["wc_co", "silicon_carbide", "sandstone"]
CARBIDE_UM = 1.36           #@param {type:"number"}
BHN_KGF_MM2 = 581.0         #@param {type:"number"}

#@markdown ### Resolution and cost
ELEMENTS_PER_DC = 5.0       #@param {type:"number"}
MICRO_GRAINS = 1            #@param {type:"integer"}
MACRO_SECTOR_MODE = "contact"  #@param ["contact", "cap"]
MACRO_GRAIN_CAP = 400000    #@param {type:"integer"}
CORES = 8                   #@param {type:"integer"}

need("SOLIDS", "cell 3")
from semgrit.sagdeck import Polyurethane, SAGParams, plan

PU = Polyurethane(c10_mpa=PU_C10_MPA, density_kg_m3=PU_DENSITY_KG_M3,
                  prony_g=PU_PRONY_G, prony_tau_s=PU_PRONY_TAU_S,
                  thickness_mm=LAYER_THICKNESS_MM)
P = SAGParams(
    diameter_mm=WHEEL_DIAMETER_MM, width_mm=WHEEL_WIDTH_MM,
    polyurethane=PU, use_shore_modulus=False,
    compression_mm=COMPRESSION_MM, speed_rpm=SPEED_RPM, friction=FRICTION,
    grain_um=float(GRAIN_UM),
    pad_areal_per_mm2=PAD_DENSITY_PER_MM2,
    material=MATERIAL, carbide_um=CARBIDE_UM, bhn_kgf_mm2=BHN_KGF_MM2,
    elements_per_dc=ELEMENTS_PER_DC, micro_grains=MICRO_GRAINS,
    macro_sector_mode=MACRO_SECTOR_MODE, macro_grain_cap=MACRO_GRAIN_CAP,
    cores=CORES, name="sag_%gum" % float(GRAIN_UM))
PLAN = plan(P)
C = PLAN["contact"]

print(chr(10).join(_sd.macro_header(PLAN)))
print("")
print(chr(10).join(_sd.micro_header(PLAN)))

## 5 · The contact, in pictures

Four things worth seeing rather than reading:

1. **Why SAG works at all** — the per-grain load against wheel compression, for
   all three pads. The collapse is the process.
2. **The patch**, with its Hertzian pressure distribution.
3. **$d_c$ three ways** — the two published geometric forms and the energy
   criterion differ by orders of magnitude on the same material, which is why
   the deck records which one it used.
4. **The regime map** — where this operating point sits relative to $d_c$.

In [ ]:
#@title 5 - The contact, drawn { display-mode: "form" }
need("PLAN", "cell 4")
import numpy as np
from semgrit import sagfig

for fn in (sagfig.load_collapse, sagfig.contact_patch,
           sagfig.dc_comparison, sagfig.regime_map):
    fn(PLAN)
    _show()

## 6 · Write the decks

Two decks, both `*Dynamic, Explicit` with **general contact**.

General contact is required here, not merely convenient, for three independent
reasons: the VUMAT **deletes elements**, and deletion exposes interior faces
that a pre-declared contact pair would never see (a chip would separate and
then pass through the tool); **which grains touch is the answer**, so it cannot
be declared in advance; and a compliant layer at high compression can fold onto
**itself**.

The MACRO deck runs three steps, and the first two are timed by the layer's own
physics rather than chosen:

| step | what it does | why that duration |
|---|---|---|
| **PRESS** | push in by $T$ | slow enough that $v/c = 0.005$ in the layer — a fast ramp loads the patch *inertially* and its pressure is not the steady Hertzian one |
| **HOLD** | dwell | $3\tau$, so the polyurethane relaxes to its **long-term** modulus, which is the state a load-cell reading and the Hertz comparison both correspond to |
| **GRIND** | rotate | the process |

In [ ]:
#@title 6 - Write MACRO and MICRO { display-mode: "form" }
WRITE_MACRO = False   #@param {type:"boolean"}
#@markdown MACRO carries the full pad, so it is ~150 MB. MICRO is the deck that
#@markdown answers the transition; leave MACRO off unless you want the contact.
OUTDIR = "RUN_SAG_NB"  #@param {type:"string"}
need("PLAN SOLIDS", "cells 3 and 4")
import os
from semgrit import sagemit

os.makedirs(OUTDIR, exist_ok=True)
MICRO = sagemit.write_micro(os.path.join(OUTDIR, "micro.inp"), PLAN, SOLIDS)
print("MICRO  %s" % MICRO["path"])
print("  %s elements, %.1f nm depth element, %.2f MB"
      % (format(MICRO["elements"], ","), MICRO["element_depth_mm"] * 1e6,
         MICRO["bytes"] / 1e6))
print("  %d passes over one track, driven by %.4e N per grain"
      % (MICRO["n_passes"], MICRO["load_per_grain_n"]))
print("  energy threshold W_p*L_c >= %.4f MPa*mm = %.1f J/m2"
      % (MICRO["energy_threshold_mpa_mm"],
         MICRO["energy_threshold_mpa_mm"] * 1000.0))
print("  dc = %.1f nm (%s)"
      % (MICRO["dc_nm"], "MEASURED" if MICRO["dc_measured"] else "computed"))

MACRO = None
if WRITE_MACRO:
    MACRO = sagemit.write_macro(os.path.join(OUTDIR, "macro.inp"), PLAN,
                                SOLIDS)
    print("")
    print("MACRO  %s" % MACRO["path"])
    print("  %s elements (%s PU, %s work), %s grains, %.1f MB"
          % (format(MACRO["elements"], ","),
             format(MACRO["pu_elements"], ","),
             format(MACRO["work_elements"], ","),
             format(MACRO["grains"], ","), MACRO["bytes"] / 1e6))
    print("  sector %.3f deg, press %.1f mm/s (v/c = %.4f)"
          % (MACRO["sector_deg"], MACRO["press_velocity_mm_s"],
             PLAN["timing"]["press_mach"]))

## 7 · Look at it — CAD, mesh, and the numbers behind both

Everything above is arithmetic. This section is where you check it by eye, and
it is the same viewer the main notebook uses — not a reduced one.

| cell | what it shows |
|---|---|
| **A1** | a *viewable* placed model of the pad |
| **A2** | the **CAD viewer** — section planes, click-to-inspect, boundary conditions, explode, colour-by-property, 12 shortcuts |
| **A3** | the **mesh viewer** — element edges, quality per part, inverted elements refused |
| **A4** | abrasive heights against the depth this process actually cuts |
| **A5** | is this a real finishing regime? measured against textbook |
| **A6** | the pad's grain distribution, as a 3-D scatter |
| **A7** | download the lot |

> **A1 needs saying plainly.** The CAD viewer draws a *placed* model — bond,
> grains, workpiece, boundary conditions. The SAG planner does not produce one:
> its "bond" is a hyperelastic ring and its grain count runs to hundreds of
> thousands. So A1 builds a rigid-wheel plan of the **same tool geometry** —
> your diameter, the pad's own measured density, the SAG depth of cut — purely
> so there is something to inspect. It is a **visualisation of the pad**, not
> the deck that gets solved. The solved decks come from cell 6.

In [ ]:
#@title A1 - A viewable model of the pad { display-mode: "form" }
#@markdown The CAD viewer draws a **placed** model: bond, grains, workpiece and
#@markdown every boundary condition the deck writes. `sagdeck.plan` does not
#@markdown produce one -- it plans the compliant two-scale model, where the
#@markdown "bond" is a hyperelastic ring and the grain count is in the hundreds
#@markdown of thousands.
#@markdown
#@markdown So this cell builds a rigid-wheel plan of the **same tool geometry**
#@markdown -- your wheel diameter, the pad's measured areal density, the SAG
#@markdown depth of cut -- so the viewer has real placed grains to show. It is a
#@markdown **visualisation of the pad**, not the deck that gets solved. The
#@markdown decks come from cell 6.
CAD_ARC_MM = 1.0        #@param {type:"number"}
CAD_WIDTH_MM = 0.30     #@param {type:"number"}
CAD_RIM_DEPTH_MM = 0.05 #@param {type:"number"}
need("PLAN SOLIDS", "cells 3 and 4")
from semgrit import materials as _materials
from semgrit.analysis import AnalysisParams
from semgrit.build_deck import DeckParams, plan_deck

_c = PLAN["contact"]
_dens = _c.active_grains / max(_c.spot_area_mm2, 1e-12)
CAD_PARAMS = DeckParams(
    name="sag_pad_view", diameter_mm=P.diameter_mm,
    include_bond=True, include_workpiece=True,
    sector_mode="arc", arc_length_mm=CAD_ARC_MM,
    rim_depth_mm=CAD_RIM_DEPTH_MM, width_mm=CAD_WIDTH_MM,
    grit_mode="areal_density", areal_density_per_mm2=_dens,
    wp_length_mm=CAD_ARC_MM * 0.2, wp_width_mm=CAD_WIDTH_MM * 0.7,
    wp_depth_mm=max(20.0 * PLAN["material"]["dc_nm"] * 1e-6, 0.005),
    wp_element_size_length_mm=CAD_ARC_MM / 100.0,
    wp_element_size_width_mm=CAD_WIDTH_MM / 100.0,
    wp_element_size_depth_mm=PLAN["micro"]["element_mm"],
    clearance_um=0.0, wp_position="centred",
    surface_speed_mm_s=_c.surface_speed_mm_s, cores=P.cores,
    analysis=AnalysisParams(
        enabled=True, depth_of_cut_um=_c.indentation_nm * 1e-3,
        material_model="hybrid",
        hybrid=_materials.hybrid_params(P.material, h_source=0, dc_form=2)))
_materials.apply(CAD_PARAMS, P.material)
CAD_PLAN = plan_deck(CAD_PARAMS, SOLIDS)
print("a viewable pad: %s grains placed on a %.0f mm tool"
      % (format(CAD_PLAN["n_grits"], ","), P.diameter_mm))
print("pad density   %.0f grains/mm2 (from the contact solution)" % _dens)
print("depth of cut  %.4f um (the per-grain indentation)"
      % (_c.indentation_nm * 1e-3))
print("")
print("This is for VIEWING. The solved decks come from cell 6.")

In [ ]:
#@title A2 - CAD viewer: the state-of-the-art one { display-mode: "form" }
#@markdown The same three.js viewer the main notebook uses, on the SAG pad.
#@markdown
#@markdown | | |
#@markdown |---|---|
#@markdown | **Shaded with edges** | feature edges over a lit surface |
#@markdown | **Wheel / Contact** | the whole 125 mm tool, or the grains on the work |
#@markdown | **Face / Axial** | straight at the pad, or down the tool axis |
#@markdown | **Section plane** | cut on any axis and drag through the model |
#@markdown | **Click a grain** | id, protrusion, height, width, volume, position |
#@markdown | **Shift-click twice** | distance and X Y Z, plus radial / along-arc / across-face |
#@markdown | **Parts tree** | show or hide the pad, the grains, the workpiece |
#@markdown | **Boundary conditions** | every symbol stands for a keyword the deck really writes |
#@markdown | **Drag block** (`G`) | drag the workpiece along the arc, shift-drag for standoff |
#@markdown | **Depth-of-cut band** | the valid window, shaded green |
#@markdown | **Colour the grains by** | protrusion, height, width, volume, or engages-the-block |
#@markdown | **Explode** | pull pad, grains and work apart along the radius |
#@markdown | **Cap the cut face** | a solid face instead of a hollow shell |
#@markdown | **Fullscreen**, **Save PNG**, **Keyboard** (`?`) | 12 shortcuts |
#@markdown
#@markdown No account, no upload. three.js loads from a CDN; the model is
#@markdown embedded in the page.
SHOW_CAD = True          #@param {type:"boolean"}
CAD_MODE = "whole wheel" #@param ["whole wheel", "wheel", "contact"]
CAD_HEIGHT = 720         #@param {type:"integer"}
CAD_MAX_INLINE_MB = 24.0 #@param {type:"number"}
need("CAD_PLAN", "cell A1")
from IPython.display import HTML, display
from semgrit.cadviewer import build as build_cad_view

if SHOW_CAD:
    _html, _meta, _info = build_cad_view(
        CAD_PLAN, os.path.join(WORK, "sag_pad.glb"), mode=CAD_MODE,
        max_grits=0, height=CAD_HEIGHT, max_inline_mb=CAD_MAX_INLINE_MB)
    print("%s: %s triangles, %d of %d grains drawn (%d in full detail)"
          % (CAD_MODE, format(_info["triangles"], ","), _meta["grits_drawn"],
             _meta["grits_total"], _meta["grits_full_detail"]))
    for _n in _meta.get("notes", []):
        print("note:", _n)
    display(HTML(_html))
else:
    print("set SHOW_CAD to draw the pad.")

In [ ]:
#@title A3 - Mesh viewer: see what will actually be solved { display-mode: "form" }
#@markdown The CAD view above is the *geometry*. This is the **mesh** -- and the
#@markdown mesh is where the arguments are.
#@markdown
#@markdown | question | how you answer it here |
#@markdown |---|---|
#@markdown | Is $d_c$ actually resolved? | the element edges are drawn; count them through the surface band |
#@markdown | Can the compliant layer **bend**? | a layer with too few elements through its thickness only shears |
#@markdown | Is anything inverted? | inverted elements are **refused**, not drawn -- Abaqus reports this as a cryptic preprocessing failure with no element numbers |
#@markdown | Is the grading where it should be? | section the block and look at the depth transition |
#@markdown
#@markdown It is the *same viewer*, fed element geometry instead of solids, so
#@markdown it keeps section capping, explode, the measuring tool and every
#@markdown shortcut. The panel is retitled for a mesh -- "click an element face"
#@markdown rather than "click a grain".
SHOW_MESH = True       #@param {type:"boolean"}
MESH_PART = "all"      #@param ["all", "tool only", "workpiece only"]
MESH_EDGES = True      #@param {type:"boolean"}
MESH_HEIGHT = 700      #@param {type:"integer"}
need("PLAN", "cell 4")
from IPython.display import HTML, display
from semgrit import meshview as _mv
from semgrit.sagwrite import build_block, build_compliant_ring

if SHOW_MESH:
    _r_out = 0.5 * P.diameter_mm
    _r_in = _r_out - P.polyurethane.thickness_mm
    _sect = min(PLAN["macro"]["sector_deg"], 30.0)
    _mic = PLAN["micro"]
    _meshes = []
    if MESH_PART in ("all", "tool only"):
        _hub = build_compliant_ring(
            inner_r_mm=max(_r_in - 2.5, 1.0), outer_r_mm=_r_in,
            width_mm=P.width_mm, sector_deg=_sect,
            n_circ=28, n_rad=2, n_axial=6)
        _pu = build_compliant_ring(
            inner_r_mm=_r_in, outer_r_mm=_r_out, width_mm=P.width_mm,
            sector_deg=_sect, n_circ=28, n_rad=6, n_axial=6)
        _meshes += [
            dict(name="hub (rigid)", nodes=_hub[0], conn=_hub[1],
                 color=_mv.C_HUB),
            dict(name="polyurethane %0.1f mm" % P.polyurethane.thickness_mm,
                 nodes=_pu[0], conn=_pu[1], color=_mv.C_COMPLIANT)]
    if MESH_PART in ("all", "workpiece only"):
        _wp = build_block(
            length_mm=_mic["side_mm"], width_mm=_mic["side_mm"],
            depth_mm=_mic["depth_mm"],
            el_length_mm=_mic["element_inplane_mm"],
            el_width_mm=_mic["element_inplane_mm"],
            fine_depth_mm=_mic["element_mm"],
            band_mm=_mic["depth_mm"] * 0.5, growth=1.3,
            x0_mm=-0.5 * _mic["side_mm"], y0_mm=-0.5 * _mic["side_mm"])
        _meshes.append(dict(name="workpiece (MICRO, dc/%g)"
                            % P.elements_per_dc,
                            nodes=_wp[0], conn=_wp[1], color=_mv.C_WORK))

    _h, _m, _i = _mv.build(_meshes, os.path.join(WORK, "sag_mesh.glb"),
                           height=MESH_HEIGHT, edges=MESH_EDGES)
    print("%-34s %10s %10s %9s %s"
          % ("part", "elements", "min edge", "aspect", "inverted"))
    for _k, _v in _m["stats"].items():
        print("%-34s %10s %9.4f nm %8.1f:1 %8d"
              % (_k[:34], format(_v["elements"], ","),
                 _v["min_edge"] * 1e6, _v["aspect_max"], _v["inverted"]))
    print("")
    print("dc = %.1f nm, surface element %.2f nm -> %.1f elements across dc"
          % (PLAN["material"]["dc_nm"], _mic["element_mm"] * 1e6,
             PLAN["material"]["dc_nm"] / (_mic["element_mm"] * 1e6)))
    for _n in _m["notes"]:
        print("note:", _n)
    display(HTML(_h))
else:
    print("set SHOW_MESH to draw the mesh.")

In [ ]:
#@title A4 - Abrasive heights, and what the pad can reach { display-mode: "form" }
#@markdown A grit cuts only as deep as it stands proud of its backing. On a
#@markdown rigid wheel that sets a hard ceiling on the depth of cut. On a SAG
#@markdown pad it matters for a different reason: the indentation is *tiny*
#@markdown against the grain, so the pad is nowhere near its geometric limit --
#@markdown and this cell shows by how much.
need("SOLIDS PLAN", "cells 3 and 4")
import numpy as _np

_h = _np.array([s.height_um for s in SOLIDS])
_c = PLAN["contact"]
_dc = PLAN["material"]["dc_nm"]
print("measured grain heights, %d solids" % len(_h))
for _q in (0, 5, 25, 50, 75, 95, 100):
    print("   %3d%%  %8.3f um" % (_q, _np.percentile(_h, _q)))
print("")
print("the pad's nominal grain size   %8.3f um" % P.grain_um)
print("mean measured height           %8.3f um" % _h.mean())
print("")
print("indentation this process makes %8.5f um  (%.3f nm)"
      % (_c.indentation_nm * 1e-3, _c.indentation_nm))
print("as a fraction of a mean grain  %8.2e" % (_c.indentation_nm * 1e-3
                                                / _h.mean()))
print("as a multiple of dc            %8.5f  (dc = %.1f nm)"
      % (_c.indentation_nm / _dc, _dc))
print("")
if _c.indentation_nm * 1e-3 < 0.01 * _h.mean():
    print("The grain is >100x deeper than the cut, so protrusion is NOT the")
    print("limit here -- which is exactly what makes SAG a finishing process")
    print("rather than a stock-removal one.")
else:
    print("The cut is a significant fraction of the grain height: check that")
    print("the pad is not being asked to cut deeper than it protrudes.")

In [ ]:
#@title A5 - Is this a real finishing regime? { display-mode: "form" }
#@markdown The deck can be geometrically perfect and still describe a process
#@markdown nobody would call grinding. These are the first questions a reviewer
#@markdown asks, and verifying the `.inp` answers none of them.
#@markdown
#@markdown **measured** rows are counted off the contact solution. **theory**
#@markdown rows are the textbook expressions for an equivalent traverse grind,
#@markdown so they need a work speed; with `WORK_SPEED_MM_MIN = 0` they are
#@markdown reported as not applicable rather than quietly computed from zero.
WORK_SPEED_MM_MIN = 15.0   #@param {type:"number"}
need("PLAN", "cell 4")
import math as _math

_c = PLAN["contact"]
_dc = PLAN["material"]["dc_nm"]
_R = 0.5 * P.diameter_mm
print("MEASURED, off the contact solution")
print("  normal load FN            %10.4f N" % _c.normal_load_n)
print("  tangential FT             %10.4f N" % (P.friction
                                                * _c.normal_load_n))
print("  spot area As              %10.2f mm2" % _c.spot_area_mm2)
print("  spot length Ls            %10.3f mm" % (2 * _c.semi_axis_a_mm))
print("  mean pressure             %10.5f MPa" % _c.mean_pressure_mpa)
print("  active grains             %10s" % format(int(_c.active_grains), ","))
print("  load per grain Fn         %10.4e N" % _c.load_per_grain_n)
print("  indentation d             %10.4f nm" % _c.indentation_nm)
print("  groove width              %10.1f nm" % _c.groove_width_nm)
print("  surface speed vs          %10.1f mm/s" % _c.surface_speed_mm_s)
print("  grain crossings / rev     %10s" % format(int(_c.grains_per_rev), ","))
print("  MRR                       %10.4f mm3/min" % _c.mrr_mm3_min)
print("")
_vw = float(WORK_SPEED_MM_MIN) / 60.0
if _vw > 0:
    print("THEORY, for an equivalent traverse grind at %.1f mm/min"
          % WORK_SPEED_MM_MIN)
    _ae = _c.indentation_nm * 1e-6
    print("  contact length sqrt(ae*de)%10.4f mm"
          % _math.sqrt(max(_ae, 0) * P.diameter_mm))
    print("  equivalent chip h_eq      %10.4e mm"
          % (_ae * _vw / max(_c.surface_speed_mm_s, 1e-9)))
    print("  speed ratio vs/vw         %10.0f"
          % (_c.surface_speed_mm_s / _vw))
    print("  removal rate Q'w          %10.4e mm3/s per mm" % (_ae * _vw))
else:
    print("THEORY: not applicable -- set WORK_SPEED_MM_MIN to compare with a")
    print("traverse grind. This is a plunge/spot configuration, and the")
    print("chip-thickness formulas need a work speed to mean anything.")
print("")
print("FINDINGS")
_bad = []
if _c.indentation_nm >= _dc:
    _bad.append("the indentation already exceeds dc, so removal is brittle "
                "from the first pass")
if _c.face_overrun > 1.0:
    _bad.append("the elliptical patch is %.1f%% wider than the %.0f mm face, "
                "so it is clipped by the wheel edges (%.1f%% of the nominal "
                "area is off the wheel)"
                % (100.0 * (_c.face_overrun - 1.0), P.width_mm,
                   100.0 * _c.area_clipped_fraction))
if not _c.density_measured:
    _bad.append("the pad density is interpolated, not measured for this "
                "grain size")
if PLAN["infeasible"]:
    _bad += list(PLAN["infeasible"])
if _bad:
    for _b in _bad:
        print("  - %s" % _b)
else:
    print("  nothing to flag: the regime is self-consistent.")

In [ ]:
#@title A6 - Quick 3-D scatter of the pad (Plotly) { display-mode: "form" }
#@markdown Every placed grain as a point, sized by protrusion. Cheaper than the
#@markdown CAD viewer and useful for seeing the *distribution* rather than the
#@markdown geometry -- whether the pad is uniform, whether the seeding clumped.
SHOW_SCATTER = True   #@param {type:"boolean"}
need("CAD_PLAN", "cell A1")
if SHOW_SCATTER:
    try:
        import plotly.graph_objects as _go
    except ImportError:
        import subprocess as _sp
        _sp.run([sys.executable, "-m", "pip", "-q", "install", "plotly"],
                check=True)
        import plotly.graph_objects as _go
    # The placement objects are on the model, not under plan["_place"] --
    # that key is a dict of per-plan arrays (baked vertices, frames, the
    # engaged set), which is a different thing entirely.
    _pl = CAD_PLAN["_model"].placements
    _x = [q.translation_mm[0] for q in _pl]
    _y = [q.translation_mm[1] for q in _pl]
    _z = [q.translation_mm[2] for q in _pl]
    _pr = [q.protrusion_mm * 1000.0 for q in _pl]
    _fig = _go.Figure(_go.Scatter3d(
        x=_x, y=_y, z=_z, mode="markers",
        marker=dict(size=3, color=_pr, colorscale="Viridis",
                    colorbar=dict(title="protrusion (um)"), opacity=0.85),
        text=["grain %d: %.2f um proud" % (i, p)
              for i, p in enumerate(_pr)]))
    _fig.update_layout(height=620, margin=dict(l=0, r=0, t=28, b=0),
                       title="%s grains on the pad, coloured by protrusion"
                             % format(len(_pl), ","),
                       scene=dict(aspectmode="data"))
    _fig.show()
else:
    print("set SHOW_SCATTER to draw it.")

## 8 · A compact mesh preview

The same viewer, fed two different things.

**The CAD** is the geometry the deck describes. **The mesh** is where the
arguments are: whether $d_c$ is actually resolved, whether the compliant layer
has enough elements through its thickness to *bend* rather than merely shear,
whether anything is inverted. Element edges are drawn, and inverted elements
are refused rather than displayed — a viewer is the last place a human looks
before submitting a multi-day job, so it is the right place to stop a mesh that
cannot run.

In [ ]:
#@title 8 - Compact mesh preview { display-mode: "form" }
SHOW = "mesh"  #@param ["mesh", "cad"]
DRAW_EDGES = True  #@param {type:"boolean"}
need("PLAN", "cell 4")
from IPython.display import HTML, display

if SHOW == "mesh":
    from semgrit import meshview as mv
    from semgrit.sagwrite import build_block, build_compliant_ring
    p = PLAN["params"]
    r_out = 0.5 * p.diameter_mm
    r_in = r_out - p.polyurethane.thickness_mm
    sect = min(PLAN["macro"]["sector_deg"], 30.0)
    hub = build_compliant_ring(inner_r_mm=r_in - 2.5, outer_r_mm=r_in,
                               width_mm=p.width_mm, sector_deg=sect,
                               n_circ=24, n_rad=2, n_axial=6)
    pu = build_compliant_ring(inner_r_mm=r_in, outer_r_mm=r_out,
                              width_mm=p.width_mm, sector_deg=sect,
                              n_circ=24, n_rad=6, n_axial=6)
    mic = PLAN["micro"]
    wp = build_block(length_mm=mic["side_mm"], width_mm=mic["side_mm"],
                     depth_mm=mic["depth_mm"],
                     el_length_mm=mic["element_inplane_mm"],
                     el_width_mm=mic["element_inplane_mm"],
                     fine_depth_mm=mic["element_mm"],
                     band_mm=mic["depth_mm"] * 0.5, growth=1.3,
                     x0_mm=-0.5 * mic["side_mm"],
                     y0_mm=-0.5 * mic["side_mm"])
    html, meta, info = mv.build(
        [dict(name="hub", nodes=hub[0], conn=hub[1], color=mv.C_HUB),
         dict(name="polyurethane", nodes=pu[0], conn=pu[1],
              color=mv.C_COMPLIANT),
         dict(name="workpiece (MICRO)", nodes=wp[0], conn=wp[1],
              color=mv.C_WORK)],
        os.path.join(WORK, "_sagmesh.glb"), height=680, edges=DRAW_EDGES)
    for k, v in meta["stats"].items():
        print("%-20s %8s elements, aspect max %6.1f:1, inverted %d"
              % (k, format(v["elements"], ","), v["aspect_max"],
                 v["inverted"]))
    for n in meta["notes"]:
        print("note:", n)
    display(HTML(html))
else:
    print("The CAD view needs a placed rigid-wheel plan; SAG's tool is")
    print("deformable, so the mesh view above IS the model. Use the")
    print("grinding-wheel notebook for the rigid-wheel CAD.")

## 9 · Verify the deck

`verify_sag_deck.py` shares **no code** with the writer. It re-parses the
`.inp` text with its own keyword-grammar reader, re-measures the node
coordinates, recomputes every hex Jacobian, and re-interprets all 58 material
constants — so a bug in the writer cannot also be baked into its own verifier.

Among the things it checks: the energy threshold recomputed from the card must
equal $H d_c$; **Bifano's $d_c$ computed from that same card** must differ, to
catch a deck that quietly fell back on the 17×-too-large value; the press must
be a *velocity* whose product with the step time equals the compression; and
the passes must **alternate direction**, because a one-way slide leaves every
point with a single pass and could never accumulate to the threshold.

In [ ]:
#@title 9 - Verify, independently { display-mode: "form" }
need("MICRO", "cell 6")
import subprocess, sys

args = [sys.executable, "verify_sag_deck.py", MICRO["path"], "--no-converge"]
if MACRO:
    args.insert(3, MACRO["path"])
r = subprocess.run(args, capture_output=True, text=True)
print(r.stdout[-9000:])
if r.stderr.strip():
    print("stderr:", r.stderr[-2000:])
print("exit code", r.returncode,
      "-- 0 means every check passed" if r.returncode == 0 else "-- SEE ABOVE")

## 10 · Mesh convergence — read this before quoting a number

The energy criterion is regularised by the element length, so it is
**mesh-dependent by construction**. Halving the element halves the work
*density* needed to trigger.

That is not a defect; it is what an energy-based failure criterion does. The
quantity the criterion actually tests, $W_p \cdot L_c$, is mesh-*independent* —
and the cell below verifies that to $10^{-16}$ while the density it corresponds
to changes fourfold.

The consequence for a paper: **$\Psi$ is calibrated for a mesh**, and any
transition depth quoted from this model has to be quoted with the element size
that produced it.

In [ ]:
#@title 10 - How much does the mesh move the answer? { display-mode: "form" }
need("PLAN", "cell 4")
import subprocess, sys
r = subprocess.run([sys.executable, "-c",
                    "import verify_sag_deck as v; v.converge()"],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr.strip():
    print("stderr:", r.stderr[-1500:])

## 11 · Rebuild the reference paper

Everything above is your process. This cell rebuilds the *paper's* experiment —
all three pads at its best operating point — so the model can be tested against
a published result.

**One parameter is calibrated, and it is worth knowing which.** The paper gives
eq. 4 for the backing pad's modulus from its shore hardness, but never prints
the shore hardness. Two independent routes exist: a hand-built CAE deck for this
process carries C10 = 0.0575 MPa ($E$ = 0.345 MPa), and inverting the contact
chain for the modulus that reproduces the paper's *stated* per-grain forces
gives 0.43 MPa. Those agree to 25 % — a real corroboration.

Pinning it tighter uses the paper's headline result (6 µm pad, pure ductile,
60–100 nm chips) together with its 30 µm force ceiling, which leaves
**C10 = 0.16606 MPa**. Only that value satisfies both constraints.

### What this can and cannot test

| testable against the paper | |
|---|---|
| contact mechanics — groove width, per-grain force, $k$ ratio | **yes**, and they land in its bands |
| **transition ordering** — 30 µm brittle → 6 µm ductile | **yes. This is the test.** |
| force magnitudes | **no** — the WC-Co Johnson-Cook constants are placeholders except $A$ |
| surface roughness $S_a$ | **no** — needs ~20 000 grain crossings against the 11–24 simulated |

**SDV13, the branch map, is the result.** Everything else is diagnostic.

In [ ]:
#@title 11 - Build the paper's three decks { display-mode: "form" }
BUILD_PAPER = False  #@param {type:"boolean"}
PADS = "all"  #@param ["all", "6 um only", "30 um only"]
#@markdown Also write run.bat / run.sh / postprocessor / EXPECTED.md per folder.
MAKE_PACKAGES = True  #@param {type:"boolean"}
import subprocess, sys

if not BUILD_PAPER:
    print("Set BUILD_PAPER to see the calibration and build the decks.")
    print("Showing the calibration only:")
    r = subprocess.run([sys.executable, "_make_sag_paper.py", "--compare"],
                       capture_output=True, text=True)
    print(r.stdout)
else:
    args = [sys.executable, "_make_sag_paper.py"]
    if PADS == "all":
        args.append("--all")
    elif PADS == "30 um only":
        args += ["--all"]
    r = subprocess.run(args, capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.stderr.strip():
        print("stderr:", r.stderr[-1500:])
    if MAKE_PACKAGES and r.returncode == 0:
        q = subprocess.run([sys.executable, "_make_sag_packages.py"],
                           capture_output=True, text=True)
        print(q.stdout[-3000:])

## 12 · Running the deck, and reading the result

```
abaqus verify -user_exp
abaqus job=micro input=micro.inp user=vumat_grind2.for double=both cpus=1 datacheck
abaqus job=micro input=micro.inp user=vumat_grind2.for double=both cpus=8 interactive
```

Or just copy a folder from `RUN_SAG/` and run its `run.bat` / `run.sh`, which
does all three in order and stops on the first failure.

**`-user_exp`, not `-user_explicit`.** The second is not an Abaqus option and
never was; it aborts the launcher before anything is submitted. A VUMAT is an
*Explicit* user subroutine, so the flag is `user_exp` — `user_std` is the
Standard equivalent, and plain `exp` verifies the *solver* rather than the
Fortran toolchain, which is the thing that actually fails on a fresh machine.
`verify_launchers.py` now checks every `run.bat` and `run.sh` in the project
against the option list Abaqus itself prints.

Three more things about that command line are not optional.

**`double=both`.** $h$ and $d_c$ are compared at 80 nm against a millimetre
geometry — a ratio of $10^{-6}$. Single precision has ~7 decimal digits and
does not have them. The failure is **silent**: the branch flag comes out wrong
and the job does not crash.

**`vumat_grind2.for`, not `vumat_grind.for`.** This deck carries 58 constants
and the energy criterion; the other subroutine reads 56 and would misinterpret
the card.

**A datacheck first.** `cpus=1 datacheck` takes seconds and reads every keyword
and the material card. The one real submission this project ever made died
exactly there, on a `*User Material` card written four values to a line instead
of eight.

### What to plot

**SDV13 is the result**: 1 = ductile, 2 = brittle.

Plot it **after every pass**, not only at the end. The criterion accumulates,
so *when* a point flips is the physics — and it is what distinguishes the three
pads from each other.

| SDV | meaning |
|---|---|
| **13** | **branch: 1 ductile, 2 brittle** |
| 14 | the chip thickness the point was given |
| 15 | $d_c$ actually used |
| 19 | strain-gradient amplification |
| 12 | deletion flag |
| 21, 22 | the energy criterion's own accumulators |

### Before quoting a force

The Johnson-Cook constants for both WC-Co and SiC are **placeholders** except
$A$, which is derived from the JH-2 card's own quasi-static compressive
strength so the two branches meet at the transition. $B, n, C, m$ and
$D_1..D_5$ are defensible orders of magnitude and nothing more.

**The branch map is the result; the force magnitudes are not**, until those are
calibrated against nanoindentation or scratch data on your own material.

In [ ]:
#@title A7 - Download everything { display-mode: "form" }
#@markdown Bundles the decks, the reports, the run scripts and the figures into
#@markdown one archive. On Colab it downloads; elsewhere it just says where the
#@markdown file is.
WHAT = "decks and reports"  #@param ["decks and reports", "everything in the output folder"]
need("MICRO", "cell 6")
import glob as _glob
import shutil as _shutil
import tarfile as _tf

_out = os.path.dirname(MICRO["path"]) or "."
_arc = os.path.join(WORK, "sag_bundle.tar.gz")
_pats = ["*.inp", "*.json", "*.csv", "*.for", "*.bat", "*.sh", "*.md",
         "*.png"] if WHAT == "decks and reports" else ["*"]
with _tf.open(_arc, "w:gz") as _t:
    _n = 0
    for _p in _pats:
        for _f in sorted(_glob.glob(os.path.join(_out, "**", _p),
                                    recursive=True)):
            if os.path.isfile(_f):
                _t.add(_f, arcname=os.path.relpath(_f, _out))
                _n += 1
print("%d file(s), %.1f MB -> %s" % (_n, os.path.getsize(_arc) / 1e6, _arc))
try:
    from google.colab import files as _files
    _files.download(_arc)
except Exception:
    print("(not Colab: copy the file from the path above)")